In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


#**Combining Dataset**
You can skip if you already have the combined dataset

In [ ]:
import os
import shutil

# Paths
vmmrdb_paths = [
    "/root/.cache/kagglehub/datasets/abhishektyagi001/vehicle-make-model-recognition-dataset-vmmrdb/versions/1/Dataset/Most_Stolen_Cars",
    "/root/.cache/kagglehub/datasets/abhishektyagi001/vehicle-make-model-recognition-dataset-vmmrdb/versions/1/Dataset/SubsetVMMR"
]
custom_dataset_path = "/content/drive/MyDrive/Gas Emission Estimation Project/cars_dataset "
combined_path = "/content/combined_cars_dataset"

def normalize_name(name):
    parts = name.split('_')
    brand = parts[0].upper()
    model = '_'.join(parts[1:])
    return brand, model

# Copy existing dataset
shutil.copytree(custom_dataset_path, combined_path, dirs_exist_ok=True)

# Integrate VMMRdb
for source in vmmrdb_paths:
    for folder in os.listdir(source):
        folder_path = os.path.join(source, folder)
        if not os.path.isdir(folder_path): continue

        brand, model = normalize_name(folder)
        dest_dir = os.path.join(combined_path, brand, model)
        os.makedirs(dest_dir, exist_ok=True)

        for file in os.listdir(folder_path):
            src = os.path.join(folder_path, file)
            dst = os.path.join(dest_dir, file)
            shutil.copy2(src, dst)

KeyboardInterrupt: 

In [ ]:
import os

total_images = 0
for brand in os.listdir(combined_path):
    brand_path = os.path.join(combined_path, brand)
    for model in os.listdir(brand_path):
        model_path = os.path.join(brand_path, model)
        images = [f for f in os.listdir(model_path) if f.endswith('.jpg')]
        total_images += len(images)

print(f"Total images after merge: {total_images}")


NotADirectoryError: [Errno 20] Not a directory: '/content/combined_cars_dataset/TOYOTA/cars_dataset.zip'

In [ ]:
from google.colab import files
files.download(output_zip + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Gas Emission Estimation Project/combined_cars_dataset.zip"
extract_path = "/content/drive/MyDrive/Gas Emission Estimation Project/combined_cars_dataset"

# Extract only if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Dataset unzipped.")
else:
    print("Dataset already exists and is ready to use.")


✅ Dataset unzipped.


or if you have the combined dataset you can just run this cell

In [2]:
dataset_dir = "/content/drive/MyDrive/Gas Emission Estimation Project/combined_cars_dataset"

#**Now, lets train Our Model on our Dataset!**




In [3]:
import os
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from tqdm import tqdm

Set Device

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Dataset and Transform

In [5]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import glob

class CarModelDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.labels = []
        self.transform = transform
        self.class_to_idx = {}
        idx = 0

        for brand in sorted(os.listdir(root_dir)):
            brand_path = os.path.join(root_dir, brand)
            if not os.path.isdir(brand_path):
                continue
            for model in sorted(os.listdir(brand_path)):
                model_path = os.path.join(brand_path, model)
                if not os.path.isdir(model_path):
                    continue
                class_name = f"{brand}_{model}"
                if class_name not in self.class_to_idx:
                    self.class_to_idx[class_name] = idx
                    idx += 1
                for img_path in glob.glob(os.path.join(model_path, "*.jpg")):
                    self.samples.append(img_path)
                    self.labels.append(self.class_to_idx[class_name])

        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self):
        return len(self.samples)



    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB")  # Ensure it's a PIL image!
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label



In [6]:
dataset_dir = "/content/drive/MyDrive/Gas Emission Estimation Project/combined_cars_dataset"

from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),  # <- Must be from torchvision.transforms
])


dataset = CarModelDataset(dataset_dir, transform=transform)
class_names = list(dataset.class_to_idx.keys())
num_classes = len(class_names)
print("Total class names: ", num_classes)

Total class names:  323


split into train/val

In [7]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

#**Model**

In [8]:
model = models.efficientnet_b0(pretrained=True)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 160MB/s]


Loss and optimizer

In [9]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Training loop

In [10]:
num_epochs = 20
train_acc, val_acc = [], []
train_loss, val_loss = [], []

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_acc = correct / total
    train_loss.append(epoch_train_loss)
    train_acc.append(epoch_train_acc)

    # Validation
    model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_running_loss / len(val_loader)
    epoch_val_acc = val_correct / val_total
    val_loss.append(epoch_val_loss)
    val_acc.append(epoch_val_acc)

    print(f"Epoch {epoch+1} | Train Acc: {epoch_train_acc:.4f}, Val Acc: {epoch_val_acc:.4f}")

Epoch 1/20: 100%|██████████| 374/374 [1:22:01<00:00, 13.16s/it]


Epoch 1 | Train Acc: 0.2246, Val Acc: 0.3444


Epoch 2/20: 100%|██████████| 374/374 [01:55<00:00,  3.24it/s]


Epoch 2 | Train Acc: 0.3448, Val Acc: 0.3798


Epoch 3/20: 100%|██████████| 374/374 [01:57<00:00,  3.18it/s]


...

Epoch 18 | Train Acc: 0.7313, Val Acc: 0.4236


Epoch 19/20: 100%|██████████| 374/374 [01:55<00:00,  3.23it/s]


Epoch 19 | Train Acc: 0.7250, Val Acc: 0.4206


Epoch 20/20: 100%|██████████| 374/374 [01:57<00:00,  3.19it/s]


Epoch 20 | Train Acc: 0.7345, Val Acc: 0.4179


In [11]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Metrics
precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')
accuracy = accuracy_score(all_labels, all_preds)

print(f"\n Final Evaluation Metrics:\nAccuracy: {accuracy:.4f}\nPrecision: {precision:.4f}\nRecall: {recall:.4f}\nF1 Score: {f1:.4f}")


 Final Evaluation Metrics:
Accuracy: 0.4179
Precision: 0.4365
Recall: 0.4179
F1 Score: 0.4118


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Next Steps:**
We can do some processing and apply some class balancing apporoaches on the datsset